In [ ]:
pip install -q pymatgen plotly smact mp_api

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.6/55.6 kB 3.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.1/5.1 MB 47.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 74.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.4/99.4 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 217.1/217.1 kB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.6/122.6 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.9/51.9 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 332.3/332.3 kB 21.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.7/117.7 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 809.0/809.0 kB 39.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 kB 4.0 MB/s eta 0:00:00
   ━━━

In [ ]:
import random
import numpy as np
from pymatgen.core import Composition, Element
import plotly.graph_objects as go

In [ ]:
elements = ["Bi", "Te", "In"]

# 1.SMACT validity

In [ ]:
import itertools
from smact.screening import smact_validity

In [ ]:
max_stoich = 8
all_compositions = [
    Composition({el: amt for el, amt in zip(elements, amt_list)}).reduced_composition
    for amt_list in itertools.product(range(max_stoich + 1), repeat=len(elements))
    if max(amt_list) > 0
]
valid_compositions = [comp for comp in all_compositions if smact_validity(comp)] #add ox threshold and see if it makes the phase diagram tighter-
unique_valid_compositions = list(set(valid_compositions))
print(
    f"Number of unique valid compositions: {len(unique_valid_compositions)} out of {len(all_compositions)}"
)

Number of unique valid compositions: 366 out of 728


In [ ]:
from smact.utils.oxidation import ICSD24OxStatesFilter

# Initialise the oxidation state filter
ox_filter = ICSD24OxStatesFilter()
ox_states_df = ox_filter.ox_states_df
ox_states_df.head(15)

,element,oxidation_state,results_count
0,H,-9,0
1,H,-8,0
2,H,-7,0
3,H,-6,0
4,H,-5,0
5,H,-4,0
6,H,-3,0
7,H,-2,0
8,H,-1,3442
9,H,0,2738


In [ ]:
supplied_oxidation_states = [
    "smact14",
    "icsd24",
    "icsd16",
    "pymatgen_sp",
    "oxidation_states_icsd24_100.txt",
]


In [ ]:
# Extract fractional compositions
e1 = np.array([comp[elements[0]] for comp in unique_valid_compositions])
e2 = np.array([comp[elements[1]] for comp in unique_valid_compositions])
e3 = np.array([comp[elements[2]] for comp in unique_valid_compositions])

# Normalize to get fractions
total = e1 + e2 + e3
e1 = e1 / total
e2 = e2 / total
e3 = e3 / total

# plot with line from origin_fig
fig = go.Figure()
# mp entries
fig.add_trace(
    go.Scatterternary(
        a=e1,
        b=e2,
        c=e3,
        mode="markers",
        marker=dict(
            size=6,
            color="green",
            symbol="circle",
            opacity=0.7,
        ),
        name="SMACT Valid",
        cliponaxis=False,
    )
)
# layout
axis_style = dict(
    title=dict(font=dict(size=10)),
    linewidth=1,
    linecolor="black",
    gridcolor="rgba(128, 128, 128, 0.2)",
    showticklabels=True,
    tickvals=[0.2, 0.4, 0.6, 0.8],
)
fig.update_layout(
    font=dict(size=10, family="Arial"),
    width=210,
    height=210,
    ternary=dict(
        bgcolor="rgba(0, 0, 0, 0)",
        aaxis=dict(axis_style, title=elements[0]),
        baxis=dict(axis_style, title=elements[1]),
        caxis=dict(axis_style, title=elements[2]),
    ),
    margin=dict(l=20, r=20, b=30, t=20),
    showlegend=False,
    legend=dict(
        x=0.65,
        y=1.08,
    ),
)
# Show the plot
fig.show()
# save image
# fig.write_image("figures/smact.pdf")

# 2. PU learning

In [ ]:
import pandas as pd

In [ ]:
max_stoich = 8
all_compositions = [
    Composition({el: amt for el, amt in zip(elements, amt_list)}).reduced_composition
    for amt_list in itertools.product(range(max_stoich + 1), repeat=len(elements))
    if max(amt_list) > 0
]
unique_composition = list(set(all_compositions))
print(
    f"Number of unique compositions: {len(unique_composition)} out of {len(all_compositions)}"
)

Number of unique compositions: 571 out of 728


In [ ]:
# Get CLscore using PU learning
# here we are just generating random data for the sake of example
pul_data = {comp: random.random() for comp in unique_composition}

In [ ]:
# Extract fractional compositions from pul_data
e1 = np.array([comp[elements[0]] for comp in pul_data.keys()])
e2 = np.array([comp[elements[1]] for comp in pul_data.keys()])
e3 = np.array([comp[elements[2]] for comp in pul_data.keys()])

# Color by CLscore
pul_values = np.array(list(pul_data.values()))

# Normalize to get fractions
total = e1 + e2 + e3
e1 = e1 / total
e2 = e2 / total
e3 = e3 / total

# plot with line from origin_fig
fig = go.Figure()
# mp entries
fig.add_trace(
    go.Scatterternary(
        a=e1,
        b=e2,
        c=e3,
        mode="markers",
        marker=dict(
            size=6,
            color="green",
            symbol="circle",
            opacity=0.7,
        ),
        name="CLscore",
        cliponaxis=False,
        marker_color=pul_values,
        marker_colorscale="Viridis",
    )
)
# layout
axis_style = dict(
    title=dict(font=dict(size=10)),
    linewidth=1,
    linecolor="black",
    gridcolor="rgba(128, 128, 128, 0.2)",
    showticklabels=True,
    tickvals=[0.2, 0.4, 0.6, 0.8],
)
fig.update_layout(
    font=dict(size=10, family="Arial"),
    width=210,
    height=210,
    ternary=dict(
        bgcolor="rgba(0, 0, 0, 0)",
        aaxis=dict(axis_style, title=elements[0]),
        baxis=dict(axis_style, title=elements[1]),
        caxis=dict(axis_style, title=elements[2]),
    ),
    margin=dict(l=20, r=20, b=30, t=20),
    showlegend=False,
    legend=dict(
        x=0.65,
        y=1.08,
    ),
)
# Show the plot
fig.show()
# save image
# fig.write_image("figures/pul.pdf")

# 3. Thermodynamics

In [ ]:
from mp_api.client import MPRester
from pymatgen.analysis.phase_diagram import PhaseDiagram, PDPlotter

In [ ]:
MP_API_KEY = ""  # TODO: change to your api
with MPRester(MP_API_KEY) as mpr:
    origin_entries = mpr.get_entries_in_chemsys(
        elements=elements, additional_criteria={"thermo_types": ["GGA_GGA+U"]}
    )
pd = PhaseDiagram(origin_entries)
pd_plotter = PDPlotter(pd)
origin_fig = pd_plotter.get_plot()
# origin_fig

Retrieving ThermoDoc documents:   0%|          | 0/68 [00:00<?, ?it/s]

In [ ]:
origin_fig.update_traces(
    mode="markers",
    cliponaxis=False,
    marker=dict(
        size=6,
        # color="green",
        # symbol="circle",
        # opacity=0.7,
    ),
)

# Set up an axis style dict (similar to your code snippet)
axis_style = dict(
    title=dict(font=dict(size=10)),
    linewidth=1,
    linecolor="black",
    gridcolor="rgba(128, 128, 128, 0.2)",
    showticklabels=True,
    tickvals=[0.2, 0.4, 0.6, 0.8],
)

# Now update layout
origin_fig.update_layout(
    font=dict(size=10, family="Arial"),
    width=210,
    height=210,
    ternary=dict(
        bgcolor="rgba(0, 0, 0, 0)",
        aaxis=dict(
            axis_style,
            title=elements[0],
        ),
        baxis=dict(
            axis_style,
            title=elements[1],
        ),
        caxis=dict(
            axis_style,
            title=elements[2],
        ),
    ),
    margin=dict(l=20, r=20, b=30, t=20),
    showlegend=False,
    legend=dict(x=0.65, y=1.08),
)

# Finally, show the updated figure
origin_fig.show()